In [1]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib symspellpy setuptools --quiet

from config import get_merged_dataframe
from main import configure

configure()

df = get_merged_dataframe(
    './data/processedNegative.csv',
    './data/processedPositive.csv',
    './data/processedNeutral.csv',
)

df.sample(10).reset_index(drop=True)

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,tweet,sentiment
0,Alright. Looks like back to chores behind the ...,negative
1,Supreme Court ropes in former top auditor to run,neutral
2,I wish but I can't sad,negative
3,Supreme Court asks Justice Karnan to explain w...,neutral
4,.the person who chose shut the fuck up: unhappy,negative
5,How govt made look benign: use revised (and lo...,neutral
6,Wow! My friend Dara shared this from her hubby...,positive
7,Nitpicking,positive
8,After being at receiving end,neutral
9,i think that's going to be true,negative


In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', message='The parameter.*token_pattern.*will not be used')

from tokenizer import (
    lemmatize_tokens,
    stem_tokens,
    snowball_stem_tokens,
    lancaster_stem_tokens,
    misspell_and_lemmatize_tokens,
    misspell_tokens
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, ComplementNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from train import train_model, evaluate_model
from vectorizer import tfidf_vectorize, count_vectorize, binary_vectorize, vectorizer_transform

config = {
    "tokenization": {
        "Tokenization": None,
        "Lemmatization": lemmatize_tokens,
        "Stemming": stem_tokens,
        "Stemming Snowball": snowball_stem_tokens,
        "Stemming Lancaster": lancaster_stem_tokens,
        "Misspellings": misspell_tokens,
        "Misspellings + Lemmatization": misspell_and_lemmatize_tokens,
    },
    "vectorization": {
        "TF-IDF": tfidf_vectorize,
        "Count": count_vectorize,
        "Binary Count": binary_vectorize,
    },
    "classifiers": [
        LogisticRegression,
        RandomForestClassifier,
        MultinomialNB,
        SVC,
        BernoulliNB,
        ComplementNB,
    ],
}

/home/samy/tweets/tokenizer.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


#### Split Data

In [3]:
# drop missing rows
df = df.dropna().reset_index(drop=True)

sentiment_column = 'sentiment'

x_train, x_test, y_train, y_test = train_test_split(
    df.drop(columns=[sentiment_column]),
    df[sentiment_column],
    test_size=0.2,
    random_state=42,
    stratify=df[sentiment_column],
)

datasets = {}

for vectorizer_name, vectorizer_func in config["vectorization"].items():
    for tokenizer_name, tokenizer_func in config["tokenization"].items():
        print(f"Generating dataset for {vectorizer_name}, {tokenizer_name}... ")
        vectorizer, x_train_bow_matrix = vectorizer_func(
            x_train,
            "tweet",
            tokenizer=tokenizer_func,
        )
        x_test_bow_matrix = vectorizer_transform(
            x_test,
            vectorizer,
            "tweet",
        )
        bow_matrix = (
            x_train_bow_matrix,
            x_test_bow_matrix,
            y_train,
            y_test,
        )
        datasets[(vectorizer_name, tokenizer_name)] = bow_matrix

Generating dataset for TF-IDF, Tokenization... 
Generating dataset for TF-IDF, Lemmatization... 
Generating dataset for TF-IDF, Stemming... 
Generating dataset for TF-IDF, Stemming Snowball... 
Generating dataset for TF-IDF, Stemming Lancaster... 
Generating dataset for TF-IDF, Misspellings... 
Generating dataset for TF-IDF, Misspellings + Lemmatization... 
Generating dataset for Count, Tokenization... 
Generating dataset for Count, Lemmatization... 
Generating dataset for Count, Stemming... 
Generating dataset for Count, Stemming Snowball... 
Generating dataset for Count, Stemming Lancaster... 
Generating dataset for Count, Misspellings... 
Generating dataset for Count, Misspellings + Lemmatization... 
Generating dataset for Binary Count, Tokenization... 
Generating dataset for Binary Count, Lemmatization... 
Generating dataset for Binary Count, Stemming... 
Generating dataset for Binary Count, Stemming Snowball... 
Generating dataset for Binary Count, Stemming Lancaster... 
Generatin

### Classification

In [4]:
results = {}

# Grid search over all combinations
for model in config['classifiers']:
    # For each vectorization technique
    model_name = model.__name__
    results[model_name] = pd.DataFrame(columns=config['vectorization'].keys(), index=config['tokenization'].keys())
    for vectorizer_name, vectorizer_func in config['vectorization'].items():
        # For each tokenization technique
        for tokenizer_name, tokenizer_func in config['tokenization'].items():
            # Print current combination
            print(f"Training {model_name}, {vectorizer_name}, {tokenizer_name}... ")
            x_train, x_test, y_train, y_test = datasets[(vectorizer_name, tokenizer_name)]
            classifier = train_model(
                x_train,
                y_train,
                classifier=model(),
            )
            acc = evaluate_model(classifier, x_test, y_test)
            # multiply by 100 to get percentage
            print(f"Accuracy: {acc:.2%}")
            results[model_name].at[tokenizer_name, vectorizer_name] = acc
            print("-" * 50)

Training LogisticRegression, TF-IDF, Tokenization... 
Accuracy: 87.57%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Lemmatization... 
Accuracy: 88.15%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming... 
Accuracy: 88.29%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Snowball... 
Accuracy: 88.58%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Stemming Lancaster... 
Accuracy: 88.44%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings... 
Accuracy: 88.44%
--------------------------------------------------
Training LogisticRegression, TF-IDF, Misspellings + Lemmatization... 
Accuracy: 87.86%
--------------------------------------------------
Training LogisticRegression, Count, Tokenization... 
Accuracy: 87.72%
--------------------------------------------------
T

In [5]:
for model in config['classifiers']:
    print(f"Results for {model.__name__}:")
    display(results[model.__name__])

Results for LogisticRegression:


,TF-IDF,Count,Binary Count
Tokenization,0.875723,0.877168,0.887283
Lemmatization,0.881503,0.881503,0.882948
Stemming,0.882948,0.877168,0.882948
Stemming Snowball,0.885838,0.877168,0.881503
Stemming Lancaster,0.884393,0.882948,0.882948
Misspellings,0.884393,0.881503,0.887283
Misspellings + Lemmatization,0.878613,0.877168,0.887283


Results for RandomForestClassifier:


,TF-IDF,Count,Binary Count
Tokenization,0.878613,0.884393,0.878613
Lemmatization,0.895954,0.875723,0.882948
Stemming,0.885838,0.877168,0.882948
Stemming Snowball,0.887283,0.869942,0.888728
Stemming Lancaster,0.887283,0.882948,0.884393
Misspellings,0.878613,0.859827,0.859827
Misspellings + Lemmatization,0.877168,0.859827,0.864162


Results for MultinomialNB:


...

,TF-IDF,Count,Binary Count
Tokenization,0.882948,0.877168,0.884393
Lemmatization,0.884393,0.878613,0.887283
Stemming,0.887283,0.880058,0.884393
Stemming Snowball,0.884393,0.880058,0.884393
Stemming Lancaster,0.880058,0.881503,0.888728
Misspellings,0.877168,0.869942,0.875723
Misspellings + Lemmatization,0.882948,0.869942,0.875723


Results for BernoulliNB:


,TF-IDF,Count,Binary Count
Tokenization,0.884393,0.884393,0.884393
Lemmatization,0.885838,0.885838,0.885838
Stemming,0.888728,0.888728,0.888728
Stemming Snowball,0.890173,0.890173,0.890173
Stemming Lancaster,0.894509,0.894509,0.894509
Misspellings,0.867052,0.867052,0.867052
Misspellings + Lemmatization,0.878613,0.878613,0.878613


Results for ComplementNB:


,TF-IDF,Count,Binary Count
Tokenization,0.84104,0.851156,0.856936
Lemmatization,0.842486,0.854046,0.854046
Stemming,0.843931,0.859827,0.859827
Stemming Snowball,0.848266,0.859827,0.859827
Stemming Lancaster,0.845376,0.854046,0.859827
Misspellings,0.83237,0.848266,0.846821
Misspellings + Lemmatization,0.82948,0.846821,0.848266
